# Experiment 2 cross-scale summary

Summarize completed Level 0 and Level 1 matched pairs without mixing their schedules or model scales.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rows=[]; layer_rows=[]
for scale in ['level0','level1']:
    state=Path(os.getenv(f'NANOGPT_EXPERIMENT2_{scale.upper()}_STATE_FILE',f'/tmp/nanogpt-experiment2-{scale}-current-pair'))
    if not state.exists():
        print('missing state:',state); continue
    pair=Path(state.read_text().strip())
    base=pair/'baseline/results'; adapt=pair/'adaptive/results'
    for arun in sorted(adapt.glob('adamw_adaptive_wwpgd_seed_*')):
        seed=int(arun.name.rsplit('_',1)[1]); brun=base/f'adamw_seed_{seed}'
        bp=brun/'selected_checkpoint_metrics.json'; ap=arun/'selected_checkpoint_metrics.json'
        if not (bp.exists() and ap.exists()): continue
        b=json.loads(bp.read_text()); a=json.loads(ap.read_text())
        summary=json.loads((arun/'controller_summary.json').read_text()) if (arun/'controller_summary.json').exists() else {}
        rows.append({'scale':scale,'seed':seed,'adamw_test_loss':b['test_loss'],'adaptive_test_loss':a['test_loss'],'test_loss_delta':a['test_loss']-b['test_loss'],'adamw_selected_step':b['selected_step'],'adaptive_selected_step':a['selected_step'],'layers_reached_target':summary.get('layers_reached_target'),'measured_layer_count':summary.get('measured_layer_count'),'projected_matrix_count':summary.get('projected_matrix_count'),'rejected_matrix_count':summary.get('rejected_matrix_count')})
        lp=arun/'layer_measurements.csv'
        if lp.exists():
            d=pd.read_csv(lp); d['scale']=scale; d['seed']=seed; layer_rows.append(d)
summary=pd.DataFrame(rows)
display(summary)


In [ ]:
if len(summary):
    aggregate=summary.groupby('scale').agg(seed_count=('seed','size'),mean_test_loss_delta=('test_loss_delta','mean'),sd_test_loss_delta=('test_loss_delta','std'),mean_layers_reached_target=('layers_reached_target','mean'),mean_projected_matrices=('projected_matrix_count','mean'),mean_rejected_matrices=('rejected_matrix_count','mean')).reset_index()
    display(aggregate)
    fig,ax=plt.subplots(figsize=(9,5))
    for scale,g in summary.groupby('scale'): ax.scatter([scale]*len(g),g.test_loss_delta,s=70,label=scale)
    ax.axhline(0,linestyle='--',linewidth=1)
    ax.set(xlabel='scale',ylabel='adaptive − AdamW selected-checkpoint test loss',title='Experiment 2 paired test-loss effects by scale')
    ax.grid(alpha=.25); plt.show()


In [ ]:
if layer_rows:
    layers=pd.concat(layer_rows,ignore_index=True)
    layers['alpha']=pd.to_numeric(layers.alpha,errors='coerce')
    final=layers.sort_values('step').groupby(['scale','seed','layer_name'],as_index=False).tail(1)
    final['in_target_band']=(final.alpha-2).abs()<=.10
    alpha_summary=final.groupby(['scale','seed']).agg(measured_layers=('alpha','count'),target_layers=('in_target_band','sum'),median_alpha=('alpha','median')).reset_index()
    alpha_summary['target_fraction']=alpha_summary.target_layers/alpha_summary.measured_layers
    display(alpha_summary)


Interpret the two scales separately first. A cross-scale pattern is descriptive until the same seed set completes at both scales.